<a href="https://colab.research.google.com/github/harsh545454/air_quality_index1/blob/main/Air_Quality_Analysis_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Air Quality Insights Dashboard - Project 4
# Works in Google Colab or locally. Needs: pandas, numpy (pre-installed in Colab)

import numpy as np
import pandas as pd

# ------------- SETTINGS (document these in your SRS) -------------
CSV_PATH = "04_air_quality.csv"
# Custom thresholds in ug/m3 - NOT an official standard. Replace with instructor's values.
BINS = [-np.inf, 30, 60, 90, 120, np.inf]
LABELS = ["Good", "Moderate", "Poor", "Very Poor", "Severe"]
HIGH_THRESHOLD = 60   # Task 14 chosen threshold
# -----------------------------------------------------------------

# ===== Task 1: Load and inspect =====
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head()) if "display" in globals() else print(df.head())
df.info()
print(df.describe().round(2))
print("Duplicate rows:", df.duplicated().sum())

# ===== Task 2: Missing values =====
print("\nMissing values per column:")
print(df.isna().sum())
print((df.isna().mean() * 100).round(2))

# ===== Task 4: Date -> datetime, extract month (done before Task 3) =====
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")   # file uses dd-mm-yyyy
df["Month"] = df["Date"].dt.month
df["Year_Month"] = df["Date"].dt.to_period("M").astype(str)  # keeps Jan-2025 and Jan-2026 apart
print("\nDate range:", df["Date"].min().date(), "to", df["Date"].max().date())

# ===== Task 3: Handle missing PM2.5 =====
# Approach: fill with each city's median.
# Why: PM2.5 is skewed with spikes (median is robust), cities differ in level
# (city-wise is fairer than one global value), and dropping rows would lose
# PM10/NO2/temperature information. Only ~2.5% is missing, so imputation is low-risk.
df["PM25_was_missing"] = df["PM25"].isna()
df["PM25"] = df["PM25"].fillna(df.groupby("City")["PM25"].transform("median"))
print("\nMissing PM2.5 after filling:", df["PM25"].isna().sum())

# ===== Task 5: Average, median, max =====
print(f"\nPM2.5 average: {df['PM25'].mean():.2f}")
print(f"PM2.5 median : {df['PM25'].median():.2f}")
print(f"PM2.5 maximum: {df['PM25'].max():.2f}")

# ===== Task 6: Rank cities by average PM2.5 =====
city_rank = (df.groupby("City")["PM25"].agg(["mean", "median", "max"])
               .round(2).sort_values("mean", ascending=False))
city_rank["Rank"] = range(1, len(city_rank) + 1)
print("\nCity ranking:\n", city_rank)

# ===== Task 7: Dates with highest PM2.5 =====
top10 = (df.sort_values("PM25", ascending=False)
           [["Date", "City", "PM25", "PM10"]].head(10))
print("\nTop 10 highest PM2.5 readings:\n", top10.to_string(index=False))

# ===== Task 8: PM10 vs PM2.5 city-wise =====
cmp_ = df.groupby("City")[["PM25", "PM10"]].mean().round(2)
cmp_["PM25/PM10 ratio"] = (cmp_["PM25"] / cmp_["PM10"]).round(2)
print("\nPM10 vs PM2.5 by city:\n", cmp_)

# ===== Task 9: Air-quality category =====
df["AQ_Category"] = pd.cut(df["PM25"], bins=BINS, labels=LABELS)
counts = df["AQ_Category"].value_counts().reindex(LABELS)
print("\nCategory counts:\n", pd.DataFrame({"count": counts,
      "percent": (counts / len(df) * 100).round(1)}))
print("\nCategory % by city:\n",
      (pd.crosstab(df["City"], df["AQ_Category"], normalize="index") * 100).round(1))

# ===== Task 10: Monthly average PM2.5 by city =====
monthly = df.groupby(["City", "Year_Month"])["PM25"].mean().round(2)
pivot = monthly.unstack("City")
print("\nMonthly average PM2.5 by city:\n", pivot)
print("\nBy calendar month (pooled):\n",
      df.groupby(["City", "Month"])["PM25"].mean().round(2).unstack("City"))
print("\nWorst month:", pivot.mean(axis=1).idxmax(),
      "| Best month:", pivot.mean(axis=1).idxmin())

# ===== Task 11: NumPy std and percentiles =====
pm = df["PM25"].to_numpy()
print(f"\nStd (ddof=1): {np.std(pm, ddof=1):.2f}")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"P{p}: {np.percentile(pm, p):.2f}")

# ===== Task 12: IQR outliers =====
q1, q3 = np.percentile(pm, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["PM25"] < low) | (df["PM25"] > high)]
print(f"\nQ1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} bounds=({low:.2f}, {high:.2f})")
print("Outlier count:", len(outliers))
print(outliers[["Date", "City", "PM25"]].to_string(index=False))

# ===== Task 13: Temperature vs PM2.5 correlation =====
print(f"\nPearson : {df['Temperature_C'].corr(df['PM25']):.3f}")
print(f"Spearman: {df['Temperature_C'].corr(df['PM25'], method='spearman'):.3f}")

# ===== Task 14: Proportion above threshold =====
above = df["PM25"] > HIGH_THRESHOLD
print(f"\nShare above {HIGH_THRESHOLD} ug/m3: {above.mean() * 100:.1f}% ({above.sum()} of {len(df)})")
print((df.groupby("City")["PM25"].apply(lambda s: (s > HIGH_THRESHOLD).mean() * 100)).round(1))

# ===== Task 15: Recommendations =====
# Write after reading your results above, e.g.:
# 1. Share of days above threshold -> year-round awareness
# 2. Top-ranked city -> targeted outreach
# 3. Worst months -> early alerts
# 4. Outlier dates -> same-day spike warnings
# 5. Correlation result -> say whether temperature is a useful signal


Shape: (1000, 7)
         Date        City   PM25   PM10   NO2  Temperature_C  Humidity_Percent
0  01-01-2025       Delhi   82.7  158.2  46.4           12.9                58
1  01-01-2025      Jaipur   97.1  161.2  45.8           13.9                62
2  01-01-2025  Chandigarh   99.7  176.2  39.4            8.4                55
3  01-01-2025     Lucknow  117.7  162.8  59.7           11.1                86
4  02-01-2025       Delhi   71.4  123.4  47.9           18.9                48
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              1000 non-null   object 
 1   City              1000 non-null   object 
 2   PM25              975 non-null    float64
 3   PM10              1000 non-null   float64
 4   NO2               1000 non-null   float64
 5   Temperature_C     1000 non-null   float64
 6   Humidity_Percent  1000 

 Public-Awareness Recommendations

1. **Run awareness campaigns all year.** About 77.1% of readings are above 60 µg/m³, and even the cleanest city, Jaipur, has 69.6% above it, so high pollution is common everywhere.

2. **Prioritise Delhi for outreach.** It has the highest average PM2.5 (86.63) and the highest share of readings above 60 µg/m³ (83.2%).

3. **Start winter alerts early.** January 2025 had the highest average PM2.5 across cities and June 2025 the lowest, so warnings should begin before the cold months.

4. **Send same-day alerts for spike days.** Six readings exceed 156.08 µg/m³, up to 182.6 in Lucknow. Spikes are sudden, so people need alerts on the day itself.

5. **Use cold weather only as a supporting cue.** The correlation between temperature and PM2.5 is -0.24, which is weak, so advice should rely on measured PM2.5 values.

*Note: the air-quality bands and the 60 µg/m³ threshold are custom choices for this project, not an official standard.*